### Load packages

In [1]:
import pandas as pd
import os
import numpy as np

### Load data files

In [2]:
# Set parameters for data files

data_directory = r"..\data\unified_csvs"
output_directory = r"..\data\merged_csvs"

vct_file = "vct_unified_prepped"
vct_path = os.path.join(data_directory, vct_file + ".csv")

output_file = "algae_merged"
output_path = os.path.join(output_directory, output_file + ".csv")

merge_config = {
    "usgs_prepped": {
        "file": "usgs_prepped.csv",
        "left_on": ["vct_report_date"],
        "right_on": ["usgs_report_date"],
        "how": "inner"
    },
    "noaa_weather_avg": {
        "file": "noaa_weather_avg.csv",
        "left_on": ["vct_report_date"],
        "right_on": ["noaa_DATE"],
        "how": "inner"
    },
    "vt_dec_unified_prepped": {
        "file": "vt_dec_unified_prepped.csv",
        "left_on": ["vct_report_date", "vct_region"],
        "right_on": ["dec_report_date", "dec_region"],
        "how": "inner"
    }
}

In [3]:
# Load VCT data file

vct_df = pd.read_csv(vct_path)

# Add vct_ prefix to feature columns if it isn't already there
vct_df.rename(
    columns=lambda col: (
        col
        if col.startswith("vct_")
        else f"vct_{col}"
    ),
    inplace=True
)

# Make sure the report_date column is a date field
vct_df["vct_report_date"] = pd.to_datetime(
    vct_df["vct_report_date"]
)

vct_df.head()

,vct_region,vct_report_date,vct_water_surface,vct_water_temp,vct_latitude,vct_longitude,vct_anabaena,vct_aphanizomenon,vct_microcystin,vct_oscillatoria,vct_target_bloom_intensity,vct_target_bloom,vct_water_temp_trailing,vct_water_surface_trailing,vct_anabaena_trailing,vct_aphanizomenon_trailing,vct_microcystin_trailing,vct_oscillatoria_trailing
0,Inland Sea,2012-07-17,NaN,NaN,44.801952,-73.257075,0,0,0,0,1a,0,NaN,NaN,NaN,NaN,NaN,NaN
1,Inland Sea,2012-07-24,NaN,NaN,44.801952,-73.257075,0,0,0,0,1a,0,NaN,NaN,0.0,0.0,0.0,0.0
2,Inland Sea,2013-06-16,Rolling,40.266667,44.801952,-73.257075,0,0,0,0,1b,0,NaN,NaN,NaN,NaN,NaN,NaN
3,Inland Sea,2013-06-17,Rolling,48.860000,44.801952,-73.257075,0,0,0,0,1b,0,40.266667,Rolling,0.0,0.0,0.0,0.0
4,Inland Sea,2013-06-18,Calm,62.000000,44.801952,-73.257075,0,0,0,0,1b,0,44.563333,Rolling,0.0,0.0,0.0,0.0


In [4]:
# Load component data files

component_dfs = {}

for file, merge_info in merge_config.items():

    print(f"Loading {file}")
    print(merge_info)

    path = os.path.join(data_directory, merge_info["file"])

    component_dfs[file] = pd.read_csv(path)

Loading usgs_prepped
{'file': 'usgs_prepped.csv', 'left_on': ['vct_report_date'], 'right_on': ['usgs_report_date'], 'how': 'inner'}
Loading noaa_weather_avg
{'file': 'noaa_weather_avg.csv', 'left_on': ['vct_report_date'], 'right_on': ['noaa_DATE'], 'how': 'inner'}
Loading vt_dec_unified_prepped
{'file': 'vt_dec_unified_prepped.csv', 'left_on': ['vct_report_date', 'vct_region'], 'right_on': ['dec_report_date', 'dec_region'], 'how': 'inner'}


### QC component data files, reformat and rename columns

In [5]:
# noaa weather data

# Add noaa_ prefix to feature columns if it isn't already there
component_dfs["noaa_weather_avg"].rename(
    columns=lambda col: (
        col
        # if col == "DATE" or col.startswith("noaa_")
        if col.startswith("noaa_")
        else f"noaa_{col}"
    ),
    inplace=True
)

# Make sure the DATE column is a date field
component_dfs["noaa_weather_avg"]["noaa_DATE"] = pd.to_datetime(
    component_dfs["noaa_weather_avg"]["noaa_DATE"]
)

component_dfs["noaa_weather_avg"].head()

,noaa_DATE,noaa_PRCP,noaa_TMIN,noaa_TMAX,noaa_AWND,noaa_WDF2,noaa_WSF2,noaa_SNWD,noaa_SNOW,noaa_WSF5,noaa_WDF5,noaa_TAVG
0,2012-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-01-02,0.50,-1.1,7.200,3.40,180.0,10.300000,0.0,0.000000,14.300000,190.000000,NaN
2,2012-01-03,1.50,-3.6,6.950,4.05,255.0,10.300000,0.0,2.500000,13.650000,265.000000,NaN
3,2012-01-04,1.00,-7.6,2.600,4.50,280.0,9.833333,0.0,3.333333,12.833333,286.666667,NaN
4,2012-01-05,0.75,-10.0,0.425,4.45,257.5,9.600000,0.0,2.500000,12.525000,260.000000,NaN


In [6]:
# USGS data

# Add usgs_ prefix to feature columns if it isn't already there
component_dfs["usgs_prepped"].rename(
    columns=lambda col: (
        col
        if col.startswith("usgs_")
        else f"usgs_{col}"
    ),
    inplace=True
)

# Make sure the report_date column is a date field
component_dfs["usgs_prepped"]["usgs_report_date"] = pd.to_datetime(
    component_dfs["usgs_prepped"]["usgs_report_date"]
)

component_dfs["usgs_prepped"].head()

,usgs_site,usgs_imputed,usgs_report_date,usgs_latitude,usgs_longitude,usgs_water_temp_max,usgs_water_temp_min,usgs_water_temp_mean,usgs_conductivity_max,usgs_conductivity_min,usgs_conductivity_mean,usgs_water_temp_max_trailing,usgs_water_temp_min_trailing,usgs_water_temp_mean_trailing,usgs_conductivity_max_trailing,usgs_conductivity_min_trailing,usgs_conductivity_mean_trailing
0,4294500,False,2014-09-30,44.47616,-73.221517,18.7,17.2,17.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4294500,False,2014-10-01,44.47616,-73.221517,17.2,15.8,16.5,179.0,170.0,173.0,18.700000,17.200000,17.900000,NaN,NaN,NaN
2,4294500,False,2014-10-02,44.47616,-73.221517,17.0,15.8,16.4,177.0,170.0,174.0,17.950000,16.500000,17.200000,179.000000,170.0,173.0
3,4294500,False,2014-10-03,44.47616,-73.221517,17.3,16.5,16.9,179.0,173.0,175.0,17.633333,16.266667,16.933333,178.000000,170.0,173.5
4,4294500,False,2014-10-04,44.47616,-73.221517,17.1,16.4,16.8,191.0,161.0,176.0,17.550000,16.325000,16.925000,178.333333,171.0,174.0


In [7]:
# DEC data

# Add dec_ prefix to feature columns if it isn't already there
component_dfs["vt_dec_unified_prepped"].rename(
    columns=lambda col: (
        col
        if col.startswith("dec_")
        else f"dec_{col}"
    ),
    inplace=True
)

# Make sure the report_date column is a date field
component_dfs["vt_dec_unified_prepped"]["dec_report_date"] = pd.to_datetime(
    component_dfs["vt_dec_unified_prepped"]["dec_report_date"]
)

component_dfs["vt_dec_unified_prepped"].head()

,dec_region,dec_report_date,dec_total nitrogen,dec_total phosphorus,dec_dissolved phosphorus,dec_chlorophyll-a,dec_secchi depth,dec_temperature
0,Main Lake Central,2012-07-01,NaN,NaN,NaN,NaN,NaN,NaN
1,Main Lake Central,2012-07-02,NaN,NaN,NaN,NaN,NaN,NaN
2,Main Lake Central,2012-07-03,NaN,NaN,NaN,NaN,NaN,NaN
3,Main Lake Central,2012-07-07,NaN,NaN,NaN,NaN,NaN,NaN
4,Main Lake Central,2012-07-08,NaN,NaN,NaN,NaN,NaN,NaN


### Merge component data files

In [8]:
# Merge components using merge_config info

algae_merged_df = vct_df.copy()

for file, merge_info in merge_config.items():
    
    print(f"Merging {file}")
    
    right_df = component_dfs[file]

    algae_merged_df = algae_merged_df.merge(
        right_df,
        left_on=merge_info.get("left_on"),
        right_on=merge_info.get("right_on"),
        # on=merge_info.get("on"),
        how=merge_info.get("how", "left")
    )

print("Rows:", len(algae_merged_df))
print("Columns:", len(algae_merged_df.columns))
print(algae_merged_df.columns)

Merging usgs_prepped
Merging noaa_weather_avg
Merging vt_dec_unified_prepped
Rows: 2919
Columns: 55
Index(['vct_region', 'vct_report_date', 'vct_water_surface', 'vct_water_temp',
       'vct_latitude', 'vct_longitude', 'vct_anabaena', 'vct_aphanizomenon',
       'vct_microcystin', 'vct_oscillatoria', 'vct_target_bloom_intensity',
       'vct_target_bloom', 'vct_water_temp_trailing',
       'vct_water_surface_trailing', 'vct_anabaena_trailing',
       'vct_aphanizomenon_trailing', 'vct_microcystin_trailing',
       'vct_oscillatoria_trailing', 'usgs_site', 'usgs_imputed',
       'usgs_report_date', 'usgs_latitude', 'usgs_longitude',
       'usgs_water_temp_max', 'usgs_water_temp_min', 'usgs_water_temp_mean',
       'usgs_conductivity_max', 'usgs_conductivity_min',
       'usgs_conductivity_mean', 'usgs_water_temp_max_trailing',
       'usgs_water_temp_min_trailing', 'usgs_water_temp_mean_trailing',
       'usgs_conductivity_max_trailing', 'usgs_conductivity_min_trailing',
       'usgs_c

### Output merged file

In [9]:
# Reorder columns so region & date are first, and target columns are last

first_cols = [c for c in ["vct_region", "vct_report_date"] if c in algae_merged_df.columns]
last_cols = [c for c in ["vct_target_bloom", "vct_target_bloom_intensity"] if c in algae_merged_df.columns]

middle_cols = [
    col for col in algae_merged_df.columns
    if col not in first_cols + last_cols
]

algae_colorder_df = algae_merged_df[first_cols + middle_cols + last_cols]

In [11]:
# Output merged data file

algae_colorder_df.to_csv(output_path, index=False)